In [1]:
from pathlib import Path
import sys
import json
import hashlib
import time

import numpy as np
import torch
import onnx

PROJECT = Path("/home/joe/PycharmProjects/id-capture")
for directory in (PROJECT, PROJECT / "training/vendor/YOLOX"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from training.yolox_workflow import make_model, show_progress
from training.yolox_export import validate_and_publish

def file_hash(path):
    with Path(path).open("rb") as source:
        return hashlib.file_digest(source, "sha256").hexdigest()

experiment = json.loads((
    PROJECT / "training/runs/yolox_geometry_experiments/latest_completed.json"
).read_text())
RUN = Path(experiment["experiment"])
CHECKPOINT = Path(experiment["checkpoints"]["Geometry"])
calibration = json.loads((RUN / "geometry_calibration.json").read_text())
assert calibration["checkpoint"] == str(CHECKPOINT)
assert file_hash(CHECKPOINT) == calibration["checkpoint_sha256"]
assert file_hash(experiment["manifest"]) == experiment["manifest_sha256"]
assert calibration["manifest_sha256"] == experiment["manifest_sha256"]

BASELINE_ONNX = PROJECT / "models/yolox-tiny-retrained.onnx"
BASELINE_ONNX_HASH = file_hash(BASELINE_ONNX)
FINAL = PROJECT / "models/yolox-tiny-geometry.onnx"
CANDIDATE = FINAL.with_name(FINAL.stem + ".candidate.onnx")
assert not FINAL.exists(), "An export already exists. Inspect it before choosing a new filename."
SIZE = experiment["size"]
print("Geometry checkpoint:", CHECKPOINT)
print("Fixed threshold:", calibration["threshold"])
print("Output:", FINAL)
print("All validation targets met:", calibration["all_targets_met"])


Geometry checkpoint: /home/joe/PycharmProjects/id-capture/training/runs/yolox_geometry_experiments/20261022_170347_473302/geometry/best.pth
Fixed threshold: 0.8
Output: /home/joe/PycharmProjects/id-capture/models/yolox-tiny-geometry.onnx
All validation targets met: False


## Export a separate candidate
Uses the same fixed-size decoded graph and runtime contract as notebook 10. Existing app model files are preserved.

In [2]:
from id_capture.onnx_detector import CONTRACT_KEY

contract = json.loads((PROJECT / "models/yolox-tiny-retrained.json").read_text())
contract.update(
    size=SIZE, input_shape=[1,3,SIZE,SIZE],
    output_shape=[1,sum((SIZE//s)**2 for s in (8,16,32)),7],
    confidence=float(calibration["threshold"]),
    checkpoint_sha256=file_hash(CHECKPOINT),
    validation_targets_met=bool(calibration["all_targets_met"]),
    experiment="geometric augmentation",
    limitation="Known passport1.jpg regression; not all validation acceptance criteria met.",
)
started = time.perf_counter()
model = make_model(CHECKPOINT).cpu().float().eval()
model.head.decode_in_inference = True
example = torch.zeros(1,3,SIZE,SIZE)
with torch.inference_mode():
    model(example)
    torch.onnx.export(
        model, example, str(CANDIDATE),
        input_names=["images"], output_names=["predictions"],
        opset_version=12, dynamic_axes=None, dynamo=False,
    )
graph = onnx.load(CANDIDATE)
metadata = {entry.key:entry.value for entry in graph.metadata_props}
metadata[CONTRACT_KEY] = json.dumps(contract)
onnx.helper.set_model_props(graph,metadata)
onnx.checker.check_model(graph)
onnx.save(graph,CANDIDATE)
bundle = dict(
    model=model, candidate=CANDIDATE, final=FINAL, contract=contract,
    manifest=Path(experiment["manifest"]), run=RUN,
)
show_progress("Candidate exported",1,1,started,finished=True)
print("Next: parity verification before finalising the export.")


Loaded pretrained tensors: 462 | New class-head tensors: 0


/tmp/ipykernel_41523/4000385075.py:19: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Candidate exported | 100.0% | Elapsed 0:00:04 | Remaining 0:00:00 | Finished 2026-10-22 18:30:23 BST
Next: parity verification before finalising the export.


## Verify conversion and save the model package
Checks raw tensors, official YOLOX postprocessing versus the runtime adapter, original-image box coordinates and final types. Uses validation samples grouped by source/class plus all phone photos. This verifies equivalence, including existing misses; it does not assert every QA image passes.

In [3]:
report = validate_and_publish(
    bundle, phone_root=PROJECT / "training/data/raw",
)
assert report["passed"]
assert file_hash(BASELINE_ONNX) == BASELINE_ONNX_HASH, "Baseline ONNX changed unexpectedly."
print("PASS:", report["images"], "parity checks")
print("Geometry ONNX:", FINAL)
print("Contract:", FINAL.with_suffix(".json"))
print("Saved threshold:", contract["confidence"])
print("Baseline app ONNX preserved. No mobile loader was changed.")


ONNX parity 1/36 | 2.8% | Elapsed 0:00:00 | Remaining 0:00:04 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 2/36 | 5.6% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 3/36 | 8.3% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 4/36 | 11.1% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 5/36 | 13.9% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 6/36 | 16.7% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 7/36 | 19.4% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 8/36 | 22.2% | Elapsed 0:00:00 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 9/36 | 25.0% | Elapsed 0:00:01 | Remaining 0:00:03 | Estimated finish 2026-10-22 18:30:27 BST
ONNX parity 10/36 | 27.8% | Elapsed 0:00:01 | Rem